In [1]:
from importlib import reload  # Python 3.4+
import pyMoM
from pyMoM import *
reload(pyMoM)
import subprocess
import os
import shutil
import pandas as pd
import re

# 20221121, acetate 0.05% (334 frames, step=12min, total 22h), (hi1, hi3, rplN, rpmB, 6300, med3, med2)
## Experimental details and data summary

In [2]:
#Experimental details

experimental_details=[{
    "date":"20221121",
    "description":"acetate",
    "f_start":"0",
    "f_end":"334",
    "condition":"acetate005",
    "t_interval":"12",
    "cell_detection_offset":"90"} #min
]

#Preprocessing infos
experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121'
##Preprocessing folder
preprocFolder="20221121_preproc"
##List of positions
#pos_var="POS_NAMES=(Pos0)"
pos_var="POS_NAMES=(Pos0 Pos1 Pos2 Pos3 Pos4 Pos5 Pos6 Pos7 Pos8 Pos9 Pos10 Pos11 Pos12 Pos13 Pos14 Pos15 Pos16 Pos17 Pos18 Pos19 Pos20 Pos21)"
##Template to use for the preprocessing
template="TEMPLATE_vng79"
##Rotation (to determine on imageJ)
ROTATION="90.1"
##TMAX, if processing is not to look at all frames
TMAX="334"
##Normalization offset
NORMALIZATION_REGION_OFFSET="120"
##Raw path to the data
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/20221121_ace0.05/20221121_ace0.05_1/20221121_ace0.05_1_MMStack.ome.tif"
##Raw path to flat field data
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221108/20221110_gfpflatfield/20221110_gfpflatfield_2/20221110_gfpflatfield_2_MMStack.ome.tif"

In [44]:
submit_preprocessing(experimentFolderPath,preprocFolder,template,NORMALIZATION_REGION_OFFSET)

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


## Moma Batch

In [3]:
#Generate a YAML file from experiment_summary_table
#Select rows of interest, that will be curated, from the curated data summary table
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20221121,7,0,0.65,24,hi1,ch
1,20221121,7,1,0.65,21,hi1,ch
2,20221121,7,2,0.65,21,hi1,ch
3,20221121,6,3,0.65,23,hi3,ch
4,20221121,6,4,0.65,24,hi3,ch
5,20221121,6,5,0.65,24,hi3,ch
6,20221121,5,6,0.65,15,rplN,ch
7,20221121,5,7,0.65,10,rplN,ch
8,20221121,5,8,0.65,15,rplN,ch
9,20221121,4,9,0.65,15,rpmB,ch


### Hi1 curation

In [18]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([0,1,2])]
print(subset)

       date  ch  pos  width  inoculated_gl promoter vector
0  20221121   7    0   0.65             24      hi1     ch
1  20221121   7    1   0.65             21      hi1     ch
2  20221121   7    2   0.65             21      hi1     ch


In [19]:
#Batch folder that should be used or created to supervise tracking/curation/export
batch_folder="batch_hi1"
batch_file="moma_batch_hi1.yaml"
create_moma_batch_directory(experimentFolderPath,batch_folder)
create_yaml_file(batch_file,experimentFolderPath,batch_folder,subset)

Batch folder already exist.
mm.properties already exists, copying cancelled.
Batch YAML file already exist. Aborting


#### Export after curation
During curation, note down comments about growth lanes curated.

In [20]:
#Final list of growth lanes that can be exported.
gls_dic={0:[3,4,7,8,14,15,18,19,20,22,24,28],
         1:[3,9,10,11,12,13,16,18,21,29],
         2:[8,9,10,12,15,16,17,18,19,20,21,25,28,29]}

write_export_yaml_file(batch_file,experimentFolderPath,batch_folder,gls_dic)

Batch YAML file already exist. Aborting


#### Tidying curated csv files for R analysis
The following will be tidied

In [21]:
print(subset,gls_dic)

       date  ch  pos  width  inoculated_gl promoter vector
0  20221121   7    0   0.65             24      hi1     ch
1  20221121   7    1   0.65             21      hi1     ch
2  20221121   7    2   0.65             21      hi1     ch {0: [3, 4, 7, 8, 14, 15, 18, 19, 20, 22, 24, 28], 1: [3, 9, 10, 11, 12, 13, 16, 18, 21, 29], 2: [8, 9, 10, 12, 15, 16, 17, 18, 19, 20, 21, 25, 28, 29]}


In [24]:
%%capture
#DO NOT MODIFY
#regular expressions for the cvs files to look for
rx0='CellStats__(.*).csv' 
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic,analysis_id=batch_file[:-5])

Copying /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/20221121_preproc/Pos0/Pos0_GL3/moma_batch_hi1/export_data__moma_batch_hi1/CellStats__20221121_ace0_Pos0_GL3.csv to /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi1_ch_curated/CellStats__20221121_ace0_Pos0_GL3.csv.

Copying /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/20221121_preproc/Pos0/Pos0_GL14/moma_batch_hi1/export_data__moma_batch_hi1/CellStats__20221121_ace0_Pos0_GL14.csv to /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi1_ch_curated/CellStats__20221121_ace0_Pos0_GL14.csv.

Copying /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/20221121_preproc/Pos0/Pos0_GL15/moma_batch_hi1/export_data__moma_batch_hi1/CellStats__20221121_ace0_Pos0_GL15.csv to /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi1_ch_curated/CellStats__20221121_ace0_Pos0_GL15.csv.

Copying /scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/20221121_preproc/Pos0/Pos0_GL18/mom

In [ ]:
#clean BKP data

In [30]:
#Generate entries for R analysis
generate_import_script_input(experimentFolderPath,subset,experiment_summary_table,experimental_details)

20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi1_ch_curated,hi1,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi3_ch_curated,hi3,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/rplN_ch_curated,rplN,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/rpmB_ch_curated,rpmB,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/6300_ch_curated,6300,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/med3_ch_curated,med3,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/med2_ch_curated,med2,ch,90


# 20220824, acetate 0.2%, (med3, hi1, rplN, 6300, rrnB, med2)
## Experimental details and data summary

In [30]:
#Experimental details
experimental_details=[{
    "date":"20220824",
    "description":"acetate",
    "f_start":"0",
    "f_end":"Inf",
    "condition":"acetate",
    "t_interval":"12",
    "cell_detection_offset":"100"} #min
]

experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824'
experimentDate=experimental_details[0]["date"]
experiment_summary_table = pd.read_csv('%s/curated_data/positions.csv'%(experimentFolderPath))

preprocFolder="20220824_ace_preproc_2"

PREPROC_DIR_TPL="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/20220824_ace_preproc_test/mmpreproc_%s"
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/20220824_ace/20220824_ace_1/20220824_ace_1_MMStack.ome.tif"
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220819_ace/20220819_flatfield_gfp_2/20220819_flatfield_gfp_2_MMStack.ome.tif"
POS_NAMES="(Pos0 Pos1 Pos2 Pos3 Pos4 Pos5 Pos6 Pos7 Pos8 Pos9 Pos10 Pos11 Pos12 Pos13 Pos14 Pos15 Pos16 Pos17 Pos18 Pos19 Pos20 Pos21 Pos22 Pos23 Pos24)"
GL_DETECTION_TEMPLATE_PATH="./TEMPLATE_vng79/template_config.json"
ROTATION="90.45"
TMAX="201"
IMAGE_REGISTRATION_METHOD="1"
FRAMES_TO_IGNORE="()"
NORMALIZATION_CONFIG_PATH="true"
NORMALIZATION_REGION_OFFSET="120"

experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20220824,4,0,1.0,3,med3,ch
1,20220824,4,1,1.0,5,med3,ch
2,20220824,4,2,1.0,3,med3,ch
3,20220824,4,3,1.0,3,med3,ch
4,20220824,6,4,1.0,25,hi1,ch
5,20220824,6,5,1.0,20,hi1,ch
6,20220824,6,6,0.8,8,hi1,ch
7,20220824,6,7,0.8,4,hi1,ch
8,20220824,7,8,1.0,34,rplN,ch
9,20220824,7,9,0.8,14,rplN,ch


In [44]:
import subprocess
subprocess.run(["./submit_slurm.sh", PREPROC_DIR_TPL, RAW_PATH, FLATFIELD_PATH, POS_NAMES, GL_DETECTION_TEMPLATE_PATH, ROTATION, TMAX, IMAGE_REGISTRATION_METHOD, FRAMES_TO_IGNORE, NORMALIZATION_CONFIG_PATH, NORMALIZATION_REGION_OFFSET])

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


### Curation med3, rplN, 6300, rrnB

In [31]:
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20220824,4,0,1.0,3,med3,ch
1,20220824,4,1,1.0,5,med3,ch
2,20220824,4,2,1.0,3,med3,ch
3,20220824,4,3,1.0,3,med3,ch
4,20220824,6,4,1.0,25,hi1,ch
5,20220824,6,5,1.0,20,hi1,ch
6,20220824,6,6,0.8,8,hi1,ch
7,20220824,6,7,0.8,4,hi1,ch
8,20220824,7,8,1.0,34,rplN,ch
9,20220824,7,9,0.8,14,rplN,ch


In [32]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([0,1,2,3,8,12,14,16,17])]
print(subset)

        date  ch  pos  width  inoculated_gl promoter vector
0   20220824   4    0    1.0              3     med3     ch
1   20220824   4    1    1.0              5     med3     ch
2   20220824   4    2    1.0              3     med3     ch
3   20220824   4    3    1.0              3     med3     ch
8   20220824   7    8    1.0             34     rplN     ch
12  20220824   2   12    1.0             14     6300     ch
14  20220824   2   14    0.8              8     6300     ch
16  20220824   1   16    1.0             16     rrnB     ch
17  20220824   1   17    1.0              9     rrnB     ch


#### Tidying curated csv files for R analysis
The following will be tidied

In [33]:
poss=list(set(list(subset['pos'])))
gls_dic={p:list(range(0,70)) for p in poss}

In [35]:
%%capture
#DO NOT MODIFY
rx0='ExportedCellStats__(.*).csv' #regular expression for the cvs files to look for
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic)

### Curation hi1

In [31]:
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20220824,4,0,1.0,3,med3,ch
1,20220824,4,1,1.0,5,med3,ch
2,20220824,4,2,1.0,3,med3,ch
3,20220824,4,3,1.0,3,med3,ch
4,20220824,6,4,1.0,25,hi1,ch
5,20220824,6,5,1.0,20,hi1,ch
6,20220824,6,6,0.8,8,hi1,ch
7,20220824,6,7,0.8,4,hi1,ch
8,20220824,7,8,1.0,34,rplN,ch
9,20220824,7,9,0.8,14,rplN,ch


In [36]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([4,5,6,7])]
print(subset)

       date  ch  pos  width  inoculated_gl promoter vector
4  20220824   6    4    1.0             25      hi1     ch
5  20220824   6    5    1.0             20      hi1     ch
6  20220824   6    6    0.8              8      hi1     ch
7  20220824   6    7    0.8              4      hi1     ch


#### Tidying curated csv files for R analysis
The following will be tidied

In [37]:
poss=list(set(list(subset['pos'])))
gls_dic={p:list(range(0,70)) for p in poss}

In [38]:
%%capture
#DO NOT MODIFY
rx0='ExportedCellStats__(.*).csv' #regular expression for the cvs files to look for
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic)

In [39]:
#Generate entries for R analysis
generate_import_script_input(experimentFolderPath,subset,experiment_summary_table,experimental_details)

20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/med3_ch_curated,med3,ch,100
20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/hi1_ch_curated,hi1,ch,100
20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/rplN_ch_curated,rplN,ch,100
20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/6300_ch_curated,6300,ch,100
20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/rrnB_ch_curated,rrnB,ch,100
20220824,acetate,0,Inf,acetate,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220824/curated_data/med2_ch_curated,med2,ch,100


# 20220921, glucose 0.2% + amino-acids (882 frames, step=1.5min, total 22h), (hi1, med2, rrnB)
## Experimental details and data summary

In [2]:
#Experimental details

experimental_details=[{
    "date":"20220921",
    "description":"glucoseaa",
    "f_start":"0",
    "f_end":"880",
    "condition":"glucoseaa",
    "t_interval":"1.5",
    "cell_detection_offset":"100"}] #min


#Preprocessing infos
experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220921'
##Preprocessing folder
preprocFolder="20220921_preproc"
##List of positions
#pos_var="POS_NAMES=(Pos0)"
pos_var="POS_NAMES=(Pos0 Pos1 Pos2 Pos3 Pos4 Pos5 Pos6 Pos7 Pos8 Pos9 Pos10 Pos11 Pos12 Pos13 Pos14 Pos15)"
##Template to use for the preprocessing
template="TEMPLATE_vng80"
##Rotation (to determine on imageJ)
ROTATION="90"
##TMAX, if processing is not to look at all frames
TMAX="880"
##Normalization offset
NORMALIZATION_REGION_OFFSET="120"
##Raw path to the data
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220921/20220921_gluaa/20220921_gluaa_3/20220921_gluaa_3_MMStack.ome.tif"
##Raw path to flat field data
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220921/20220921_gluaa_gfpflatfield_2/20220921_gluaa_gfpflatfield_2_MMStack.ome.tif"

In [44]:
submit_preprocessing(experimentFolderPath,preprocFolder,template,NORMALIZATION_REGION_OFFSET)

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


## Moma Batch

In [3]:
#Generate a YAML file from experiment_summary_table
#Select rows of interest, that will be curated, from the curated data summary table
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20220921,7,0,1.25,1,hi1,ch
1,20220921,7,1,1.25,1,hi1,ch
2,20220921,7,2,1.25,1,hi1,ch
3,20220921,4,3,1.25,1,med2,ch
4,20220921,4,4,1.25,1,med2,ch
5,20220921,4,5,1.25,1,med2,ch
6,20220921,4,6,1.25,1,med2,ch
7,20220921,4,7,1.25,1,med2,ch
8,20220921,4,8,1.25,1,med2,ch
9,20220921,4,9,1.25,1,med2,ch


### Hi1 curation

In [4]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([0,1])]
print(subset)

       date  ch  pos  width  inoculated_gl promoter vector
0  20220921   7    0   1.25              1      hi1     ch
1  20220921   7    1   1.25              1      hi1     ch


In [5]:
#Batch folder that should be used or created to supervise tracking/curation/export
batch_folder="batch_hi1"
batch_file="moma_batch_hi1.yaml"
create_moma_batch_directory(experimentFolderPath,batch_folder)
create_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,subset)

Batch folder already exist.
mm.properties already exists, copying cancelled.
/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20220921/20220921_preproc
Batch YAML file already exist. Aborting


#### Export after curation
During curation, note down comments about growth lanes curated.

In [6]:
#Final list of growth lanes that can be exported.
gls_dic={0:[5,7,8,11,12,17,19,28,36,40,42,43,48,50,51,53],
         1:[2,5,7,8,11,12,13,15,17,18,21,22,23,24,25,26,33,36,40,42,43,44,45,47,48,51,52,54,55,57,58]}

write_export_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,gls_dic)

Batch YAML file already exist. Aborting


#### Tidying curated csv files for R analysis
The following will be tidied

In [7]:
print(subset,gls_dic)

       date  ch  pos  width  inoculated_gl promoter vector
0  20220921   7    0   1.25              1      hi1     ch
1  20220921   7    1   1.25              1      hi1     ch {0: [5, 7, 8, 11, 12, 17, 19, 28, 36, 40, 42, 43, 48, 50, 51, 53], 1: [2, 5, 7, 8, 11, 12, 13, 15, 17, 18, 21, 22, 23, 24, 25, 26, 33, 36, 40, 42, 43, 44, 45, 47, 48, 51, 52, 54, 55, 57, 58]}


In [8]:
%%capture
#DO NOT MODIFY
#regular expressions for the cvs files to look for
rx0='CellStats__(.*).csv' 
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic,analysis_id='moma_batch_hi1_2')

In [28]:
batch_file

'moma_batch_hi1.yaml'

In [ ]:
#clean BKP data

In [30]:
#Generate entries for R analysis
generate_import_script_input(experimentFolderPath,subset,experiment_summary_table,experimental_details)

20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi1_ch_curated,hi1,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/hi3_ch_curated,hi3,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/rplN_ch_curated,rplN,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/rpmB_ch_curated,rpmB,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/6300_ch_curated,6300,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/med3_ch_curated,med3,ch,90
20221121,acetate,0,334,acetate005,12,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20221121/curated_data/med2_ch_curated,med2,ch,90


# 20190515, glucose 0.2% (400 frames, step=3min, total 20h), glycerol 0.4% (440 frames, step=6 min, 40h), (hi1, med1, med2, rpmB)

## Part of the data analyzed using the batch mode, growth lanes opening towards the right (hi1)
### Experimental details and data summary

In [34]:
#Experimental details

experimental_details=[{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"0",
    "f_end":"440",
    "condition":"glucose",
    "t_interval":"3",
    "cell_detection_offset":"100"} #min
    ,
{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"440",
    "f_end":"Inf",
    "condition":"glycerol",
    "t_interval":"6",
    "cell_detection_offset":"100"} #min
]

#Preprocessing infos
experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515'
##Preprocessing folder
preprocFolder="20190515_preproc_right_bis"
##List of positions
#pos_var="POS_NAMES=(Pos0)"
pos_var="POS_NAMES=(Pos0 Pos1 Pos2 Pos3 Pos4 Pos5 Pos6 Pos9 Pos10 Pos12 Pos13 Pos14 Pos15 Pos16)"
##Template to use for the preprocessing
template="TEMPLATE_vng20_right"
##Rotation (to determine on imageJ)
ROTATION="-0.5"
##TMAX, if processing is not to look at all frames
TMAX="881"
##Normalization offset
NORMALIZATION_REGION_OFFSET="120"
##Raw path to the data
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_hi1_med1_med2_rpmB_glu_gly_7/20190515_hi1_med1_med2_rpmB_glu_gly_7_MMStack.ome.tif"
##Raw path to flat field data
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_flatfieldGFP/20190515_flatfieldGFP_2_MMStack.ome.tif"

In [44]:
submit_preprocessing(experimentFolderPath,preprocFolder,template,NORMALIZATION_REGION_OFFSET)

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


## Moma Batch

In [35]:
#Generate a YAML file from experiment_summary_table
#Select rows of interest, that will be curated, from the curated data summary table
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20190515,7,0,0.8,full,hi1,ch
1,20190515,7,1,0.8,full,hi1,ch
2,20190515,7,2,0.8,full,hi1,ch
3,20190515,7,3,0.8,full,hi1,ch
4,20190515,7,4,0.8,full,hi1,ch
5,20190515,7,5,0.8,full,hi1,ch
6,20190515,5,6,0.8,7,med1,ch
7,20190515,5,9,0.8,7,med1,ch
8,20190515,5,10,0.8,8,med1,ch
9,20190515,5,12,0.8,8,med1,ch


### Hi1 curation

In [36]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([0,1])]
print(subset)

       date  ch  pos  width inoculated_gl promoter vector
0  20190515   7    0    0.8          full      hi1     ch
1  20190515   7    1    0.8          full      hi1     ch


In [38]:
#Batch folder that should be used or created to supervise tracking/curation/export
batch_folder="batch_hi1"
batch_file="moma_batch_hi1.yaml"
create_moma_batch_directory(experimentFolderPath,batch_folder)
create_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,subset)

Batch folder already exist.
mm.properties already exists, copying cancelled.
/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_preproc_right_bis
Batch file does not exist. It is created for the following positions, for all detected growth lanes:
Pos0
Pos1


In [39]:
preproc_dir="%s/%s"%(experimentFolderPath,preprocFolder)
preproc_dir

'/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_preproc_right_bis'

#### Export after curation
During curation, note down comments about growth lanes curated.

In [40]:
#Final list of growth lanes that can be exported.
gls_dic={0:[2,3,4,6,8,9,10,12,13,14,16,17,18,19],
         1:[1,2,3,4,5,6,8,9,10,11,12]}

write_export_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,gls_dic)

Batch file does not exist. It is created for the following positions, for all curated growth lanes in the export dictionnary:
Pos0
Pos1


#### Tidying curated csv files for R analysis
The following will be tidied

In [68]:
print(subset,gls_dic)

       date  ch  pos  width inoculated_gl promoter vector
0  20190515   7    0    0.8          full      hi1     ch
1  20190515   7    1    0.8          full      hi1     ch {0: [2, 3, 4, 6, 8, 9, 10, 12, 13, 14, 16, 17, 18, 19], 1: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12]}


In [69]:
%%capture
#DO NOT MODIFY
#regular expressions for the cvs files to look for
rx0='CellStats__(.*).csv' 
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic,analysis_id='moma_batch_hi1')

In [70]:
#Generate entries for R analysis
generate_import_script_input(experimentFolderPath,subset,experiment_summary_table,experimental_details)

20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/hi1_ch_curated,hi1,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med1_ch_curated,med1,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med2_ch_curated,med2,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/rpmB_ch_curated,rpmB,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/hi1_ch_curated,hi1,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med1_ch_curated,med1,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med2_ch_curated

## Part of the data analyzed using the batch mode, growth lanes opening towards the left (med2, rpmB)
### Experimental details and data summary

In [8]:
#Experimental details

experimental_details=[{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"0",
    "f_end":"440",
    "condition":"glucose",
    "t_interval":"3",
    "cell_detection_offset":"100"} #min
    ,
{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"440",
    "f_end":"Inf",
    "condition":"glycerol",
    "t_interval":"6",
    "cell_detection_offset":"100"} #min
]

#Preprocessing infos
experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515'
##Preprocessing folder
preprocFolder="20190515_preproc_left"
##List of positions
#pos_var="POS_NAMES=(Pos0)"
pos_var="POS_NAMES=(Pos17 Pos18 Pos19 Pos20 Pos21 Pos22 Pos23 Pos24 Pos25 Pos26 Pos27 Pos28)"
##Template to use for the preprocessing
template="TEMPLATE_vng20_left"
##Rotation (to determine on imageJ)
ROTATION="-0.5"
##TMAX, if processing is not to look at all frames
TMAX="881"
##Normalization offset
NORMALIZATION_REGION_OFFSET="120"
##Raw path to the data
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_hi1_med1_med2_rpmB_glu_gly_7/20190515_hi1_med1_med2_rpmB_glu_gly_7_MMStack.ome.tif"
##Raw path to flat field data
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_flatfieldGFP/20190515_flatfieldGFP_2_MMStack.ome.tif"

In [44]:
submit_preprocessing(experimentFolderPath,preprocFolder,template,NORMALIZATION_REGION_OFFSET)

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


## Moma Batch

In [9]:
#Generate a YAML file from experiment_summary_table
#Select rows of interest, that will be curated, from the curated data summary table
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20190515,7,0,0.8,full,hi1,ch
1,20190515,7,1,0.8,full,hi1,ch
2,20190515,7,2,0.8,full,hi1,ch
3,20190515,7,3,0.8,full,hi1,ch
4,20190515,7,4,0.8,full,hi1,ch
5,20190515,7,5,0.8,full,hi1,ch
6,20190515,5,6,0.8,7,med1,ch
7,20190515,5,9,0.8,7,med1,ch
8,20190515,5,10,0.8,8,med1,ch
9,20190515,5,12,0.8,8,med1,ch


### Hi1 curation

In [11]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([19,25])]
print(subset)

        date  ch  pos  width inoculated_gl promoter vector
16  20190515   4   19    0.8            15     med2     ch
22  20190515   2   25    0.8          full     rpmB     ch


In [12]:
#Batch folder that should be used or created to supervise tracking/curation/export
batch_folder="batch_med2_rpmB"
batch_file="moma_batch_pos_19_25.yaml"
create_moma_batch_directory(experimentFolderPath,batch_folder)
create_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,subset)

Batch folder already exist.
mm.properties already exists, copying cancelled.
/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_preproc_left
Batch YAML file already exist. Aborting


#### Export after curation
During curation, note down comments about growth lanes curated.

In [13]:
#Final list of growth lanes that can be exported.
gls_dic={19:[2,4,5,7,8,9,10,12,13,14,15,16,17],
         25:[1,2,3,4,5,6,7,8,11,12,14,15,17,18]}

write_export_yaml_file(batch_file,experimentFolderPath,preprocFolder,batch_folder,gls_dic)

Batch YAML file already exist. Aborting


#### Tidying curated csv files for R analysis
The following will be tidied

In [14]:
print(subset,gls_dic)

        date  ch  pos  width inoculated_gl promoter vector
16  20190515   4   19    0.8            15     med2     ch
22  20190515   2   25    0.8          full     rpmB     ch {19: [2, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17], 25: [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 18]}


In [69]:
%%capture
#DO NOT MODIFY
#regular expressions for the cvs files to look for
rx0='CellStats__(.*).csv' 
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic,analysis_id='moma_batch_hi1')

In [70]:
#Generate entries for R analysis
generate_import_script_input(experimentFolderPath,subset,experiment_summary_table,experimental_details)

20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/hi1_ch_curated,hi1,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med1_ch_curated,med1,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med2_ch_curated,med2,ch,100
20190515,glucose_glycerol_switch,0,440,glucose,3,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/rpmB_ch_curated,rpmB,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/hi1_ch_curated,hi1,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med1_ch_curated,med1,ch,100
20190515,glucose_glycerol_switch,440,Inf,glycerol,6,/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/curated_data/med2_ch_curated

## Part of the data that were previously analyzed NOT using the batch mode, growth lanes opening to the right

In [51]:
#Experimental details

experimental_details=[{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"0",
    "f_end":"440",
    "condition":"glucose",
    "t_interval":"3",
    "cell_detection_offset":"100"} #min
    ,
{
    "date":"20190515",
    "description":"glucose_glycerol_switch",
    "f_start":"440",
    "f_end":"Inf",
    "condition":"glycerol",
    "t_interval":"6",
    "cell_detection_offset":"100"} #min
]

#Preprocessing infos
experimentFolderPath='/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515'
##Preprocessing folder
preprocFolder="20190515_preproc_right_bis"
##List of positions
#pos_var="POS_NAMES=(Pos0)"
pos_var="POS_NAMES=(Pos0 Pos1 Pos2 Pos3 Pos4 Pos5 Pos6 Pos9 Pos10 Pos12 Pos13 Pos14 Pos15 Pos16)"
##Template to use for the preprocessing
template="TEMPLATE_vng20_right"
##Rotation (to determine on imageJ)
ROTATION="-0.5"
##TMAX, if processing is not to look at all frames
TMAX="881"
##Normalization offset
NORMALIZATION_REGION_OFFSET="120"
##Raw path to the data
RAW_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_hi1_med1_med2_rpmB_glu_gly_7/20190515_hi1_med1_med2_rpmB_glu_gly_7_MMStack.ome.tif"
##Raw path to flat field data
FLATFIELD_PATH="/scicore/home/nimwegen/rocasu25/MM_Data/Dany/20190515/20190515_flatfieldGFP/20190515_flatfieldGFP_2_MMStack.ome.tif"

In [ ]:
import subprocess
subprocess.run(["./submit_slurm.sh", PREPROC_DIR_TPL, RAW_PATH, FLATFIELD_PATH, POS_NAMES, GL_DETECTION_TEMPLATE_PATH, ROTATION, TMAX, IMAGE_REGISTRATION_METHOD, FRAMES_TO_IGNORE, NORMALIZATION_CONFIG_PATH, NORMALIZATION_REGION_OFFSET])

Do you really want to submit preprocessing tasks to the scheduler? If the preprocessing folder target path has not been changed, it might result in overwritting your data. Enter 'Yes', to confirm. no


Aborting scheduler preprocessing submission


### Curation hi1

In [53]:
experiment_summary_table=print_curated_data_summary(experimental_details,experimentFolderPath)
experiment_summary_table

,date,ch,pos,width,inoculated_gl,promoter,vector
0,20190515,7,0,0.8,full,hi1,ch
1,20190515,7,1,0.8,full,hi1,ch
2,20190515,7,2,0.8,full,hi1,ch
3,20190515,7,3,0.8,full,hi1,ch
4,20190515,7,4,0.8,full,hi1,ch
5,20190515,7,5,0.8,full,hi1,ch
6,20190515,5,6,0.8,7,med1,ch
7,20190515,5,9,0.8,7,med1,ch
8,20190515,5,10,0.8,8,med1,ch
9,20190515,5,12,0.8,8,med1,ch


In [54]:
subset=experiment_summary_table[experiment_summary_table["pos"].isin([3,4,5])]
print(subset)

       date  ch  pos  width inoculated_gl promoter vector
3  20190515   7    3    0.8          full      hi1     ch
4  20190515   7    4    0.8          full      hi1     ch
5  20190515   7    5    0.8          full      hi1     ch


#### Tidying curated csv files for R analysis
The following will be tidied

In [55]:
poss=list(set(list(subset['pos'])))
gls_dic={p:list(range(0,70)) for p in poss}

In [56]:
%%capture
#DO NOT MODIFY
rx0='ExportedCellStats__(.*).csv' #regular expression for the cvs files to look for
pyMoM.extractAndTidyCurated(experimentFolderPath,preprocFolder,subset,rx0,gls_dic)